In [1]:
import pandas as pd
import numpy as np
import optuna
import warnings
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from optuna.integration import LightGBMPruningCallback
import xgboost as xgb
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#importing dataset
raw_data = fetch_california_housing()
data = pd.DataFrame(raw_data.data, columns=raw_data.feature_names)
data['MedHouseVal'] = raw_data.target

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


In [3]:
#remove outlier
numeric_cols = data.select_dtypes(include=["float", "int"]).columns.tolist()

for col in numeric_cols:
    Q1 = data[col].quantile(0.3)
    Q3 = data[col].quantile(0.7)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    data[col] = np.where(data[col].between(lower, upper), data[col], np.nan)

data = data.dropna()

#combined features
data['Ave_Room_Bed'] = data['AveRooms'] / data['AveBedrms']
data['HouseRooms'] = data['HouseAge'] / data['AveRooms']
data['Rooms_per_Person'] = data['AveRooms'] / data['AveOccup']

In [4]:
'''
def objective(trial):
    # Internal split for validation during tuning
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42
    )
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    # Search space
    param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',  # Metric for early stopping and pruning
        'tree_method': 'hist',
        
        # --- Structure ---
        # California has complex interactions, we allow deeper trees
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        # Important to avoid overfitting on price outliers
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
        'gamma': trial.suggest_float('gamma', 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regularization ---
        # Crucial in regression to avoid chasing extreme prices
        'lambda': trial.suggest_float('lambda', 1e-9, 15.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-9, 15.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.6),
    }

    # Pruning based on RMSE
    pruning_callback = optuna.integration.XGBoostPruningCallback(
        trial, "validation-rmse"
    )

    model = xgb.train(
        param,
        dtrain,
        num_boost_round=1000,
        evals=[(dvalid, "validation")],
        early_stopping_rounds=100,
        callbacks=[pruning_callback],
        verbose_eval=False
    )

    # Optuna must minimize RMSE
    preds = model.predict(dvalid)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    return rmse
'''


'\ndef objective(trial):\n    # Internal split for validation during tuning\n    X_train, X_valid, y_train, y_valid = train_test_split(\n        X_train_full, y_train_full, test_size=0.2, random_state=42\n    )\n\n    dtrain = xgb.DMatrix(X_train, label=y_train)\n    dvalid = xgb.DMatrix(X_valid, label=y_valid)\n\n    # Search space\n    param = {\n        \'verbosity\': 0,\n        \'objective\': \'reg:squarederror\',\n        \'eval_metric\': \'rmse\',  # Metric for early stopping and pruning\n        \'tree_method\': \'hist\',\n\n        # --- Structure ---\n        # California has complex interactions, we allow deeper trees\n        \'max_depth\': trial.suggest_int(\'max_depth\', 2, 20),\n        # Important to avoid overfitting on price outliers\n        \'min_child_weight\': trial.suggest_int(\'min_child_weight\', 1, 25),\n        \'gamma\': trial.suggest_float(\'gamma\', 1e-7, 1.5, log=True),\n\n        # --- Randomness ---\n        \'subsample\': trial.suggest_float(\'subsampl

In [5]:
def objective(trial):
    # Internal split for validation during tuning
    X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    # Search space
    param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse', # Metric for early stopping and pruning
        'tree_method': 'hist',
        
        # --- Structure ---
        # California has complex interactions, we allow deeper trees
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        # Important to avoid overfitting on price outliers
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
        'gamma': trial.suggest_float('gamma', 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regularization ---
        # Fundamental in regression to avoid chasing extreme prices
        'lambda': trial.suggest_float('lambda', 1e-9, 15.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-9, 15.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.6),
    }

    model = xgb.cv(
        params=param,
        dtrain=dtrain,
        num_boost_round=3000,
        nfold=5,
        early_stopping_rounds=100,
        verbose_eval=False
    )

    best_rmse = model["test-rmse-mean"].iloc[-1]
    return best_rmse

In [6]:
# X, y = data.data, data.target
X = data.drop(columns=["MedHouseVal"])
y = data["MedHouseVal"]

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset Shape: {X.shape}")
print(f"Average target: {np.mean(y):.2f} (hundreds of k$)")

# ==========================================
# 2. Baseline (Without Tuning)
# ==========================================
print("\n--- Training Baseline Model (Default) ---")
dtrain_full = xgb.DMatrix(X_train_full, label=y_train_full)
dtest = xgb.DMatrix(X_test, label=y_test)

# Default standard parameters
params_base = {
    'objective': 'reg:squarederror',  # Regression
    'tree_method': 'hist',
    'random_state': 42
}

model_base = xgb.train(params_base, dtrain_full, num_boost_round=100)
preds_base = model_base.predict(dtest)

# Compute Baseline RMSE
rmse_base = np.sqrt(mean_squared_error(y_test, preds_base))
print(f"Baseline RMSE: {rmse_base:.4f}")


# ==========================================
# 3. Optimization with Optuna
# ==========================================
print("\n--- Starting Tuning with Optuna ---")

# Create Study (Minimize RMSE)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=200, timeout=600)

print(f"\nBest RMSE found by Optuna (Validation): {study.best_value:.4f}")
print("Best parameters:", study.best_params)


# ==========================================
# 4. Final Model Training (Best Params + Shrinkage)
# ==========================================
print("\n--- Training Final Optimized Model ---")

best_params = study.best_params
best_params['objective'] = 'reg:squarederror'
best_params['eval_metric'] = 'rmse'
best_params['tree_method'] = 'hist'

# "SHRINKAGE" STRATEGY (Refining gradient descent)
# Lower the found learning rate for better final precision
best_params['learning_rate'] = best_params['learning_rate'] * 0.5
# Increase the number of trees accordingly (high safety buffer, early stopping will handle the rest)
num_round_final = 5000

model_opt = xgb.train(
    best_params,
    dtrain_full,
    num_boost_round=num_round_final,
    evals=[(dtest, "test")],  # Test set used ONLY to stop training at the right point
    early_stopping_rounds=100,
    verbose_eval=False
)

# ==========================================
# 5. Results Analysis
# ==========================================
print("\n--- FINAL COMPARISON ---")

preds_opt = model_opt.predict(dtest)
rmse_opt = np.sqrt(mean_squared_error(y_test, preds_opt))

# R2 Score (coefficient of determination)
r2_base = r2_score(y_test, preds_base)
r2_opt = r2_score(y_test, preds_opt)

print(f"RMSE Baseline:    {rmse_base:.4f}")
print(f"RMSE Optimized:   {rmse_opt:.4f}")
print(f"--> Error Improvement: {rmse_base - rmse_opt:.4f} (lower is better)")
print("-" * 30)
print(f"R2 Score Baseline:    {r2_base:.4f}")
print(f"R2 Score Optimized:   {r2_opt:.4f}")
print(f"--> Extra Explained Variance: +{(r2_opt - r2_base)*100:.2f}%")

Dataset Shape: (13561, 11)
Average target: 1.85 (hundreds of k$)

--- Training Baseline Model (Default) ---


[I 2026-03-27 22:24:35,237] A new study created in memory with name: no-name-ef402601-f35f-49cc-a999-1fcce7483132


Baseline RMSE: 0.3548

--- Starting Tuning with Optuna ---


[I 2026-03-27 22:24:36,371] Trial 0 finished with value: 0.38860288527476416 and parameters: {'max_depth': 12, 'min_child_weight': 17, 'gamma': 0.09662772887542037, 'subsample': 0.9264462158902, 'colsample_bytree': 0.7981293407383296, 'lambda': 2.582542097270892e-07, 'alpha': 0.005981059824203757, 'learning_rate': 0.36108702961552114}. Best is trial 0 with value: 0.38860288527476416.
[I 2026-03-27 22:24:43,419] Trial 1 finished with value: 0.35809275252933315 and parameters: {'max_depth': 7, 'min_child_weight': 17, 'gamma': 1.4760201038619426e-05, 'subsample': 0.6671552968534517, 'colsample_bytree': 0.6714899557816904, 'lambda': 4.864380430394951, 'alpha': 1.927422347042258, 'learning_rate': 0.07836498900565643}. Best is trial 1 with value: 0.35809275252933315.
[I 2026-03-27 22:24:55,999] Trial 2 finished with value: 0.38321537328638555 and parameters: {'max_depth': 19, 'min_child_weight': 6, 'gamma': 1.255572632953403e-06, 'subsample': 0.9909225031394892, 'colsample_bytree': 0.8321495


Best RMSE found by Optuna (Validation): 0.3525
Best parameters: {'max_depth': 15, 'min_child_weight': 19, 'gamma': 6.636602209231141e-06, 'subsample': 0.7774625168288367, 'colsample_bytree': 0.6629831522724101, 'lambda': 3.488805072603698e-07, 'alpha': 1.6352743946996894e-07, 'learning_rate': 0.005471447310814129}

--- Training Final Optimized Model ---

--- FINAL COMPARISON ---
RMSE Baseline:    0.3548
RMSE Optimized:   0.3342
--> Error Improvement: 0.0206 (lower is better)
------------------------------
R2 Score Baseline:    0.8194
R2 Score Optimized:   0.8398
--> Extra Explained Variance: +2.03%


In [7]:
'''
def objective(trial):
    # Internal split for validation during tuning
    X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    # Search space
    param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse', # Metric for early stopping and pruning
        'tree_method': 'hist',
        
        # --- Structure ---
        # California has complex interactions, we allow deeper trees
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        # Important to avoid overfitting on price outliers
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
        'gamma': trial.suggest_float('gamma', 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regularization ---
        # Fundamental in regression to avoid chasing extreme prices
        'lambda': trial.suggest_float('lambda', 1e-9, 15.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-9, 15.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.6),
    }

    # Pruning based on RMSE
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "validation-rmse")

    model = xgb.train(
        param,
        dtrain_pseudo,
        num_boost_round=1000,
        evals=[(dvalid, "validation")],
        early_stopping_rounds=100,
        callbacks=[pruning_callback],
        verbose_eval=False
    )

    # Optuna must minimize RMSE
    preds = model.predict(dvalid)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    return rmse
'''

'\ndef objective(trial):\n    # Internal split for validation during tuning\n    X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)\n\n    dtrain = xgb.DMatrix(X_train, label=y_train)\n    dvalid = xgb.DMatrix(X_valid, label=y_valid)\n\n    # Search space\n    param = {\n        \'verbosity\': 0,\n        \'objective\': \'reg:squarederror\',\n        \'eval_metric\': \'rmse\', # Metric for early stopping and pruning\n        \'tree_method\': \'hist\',\n\n        # --- Structure ---\n        # California has complex interactions, we allow deeper trees\n        \'max_depth\': trial.suggest_int(\'max_depth\', 2, 20),\n        # Important to avoid overfitting on price outliers\n        \'min_child_weight\': trial.suggest_int(\'min_child_weight\', 1, 25),\n        \'gamma\': trial.suggest_float(\'gamma\', 1e-7, 1.5, log=True),\n\n        # --- Randomness ---\n        \'subsample\': trial.suggest_float(\'subsample\', 0.6, 1.0),\n

In [8]:
def objective(trial):
    # Internal split for validation during tuning
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42
    )
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)

    # Search space
    param = {
        'verbosity': 0,
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',  # Metric for early stopping and pruning
        'tree_method': 'hist',
        
        # --- Structure ---
        # California has complex interactions, so we allow deeper trees
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        # Important to avoid overfitting on price outliers
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 25),
        'gamma': trial.suggest_float('gamma', 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        
        # --- Regularization ---
        # Fundamental in regression to avoid chasing extreme prices
        'lambda': trial.suggest_float('lambda', 1e-9, 15.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-9, 15.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.6),
    }

    model = xgb.cv(
        params=param,
        dtrain=dtrain_pseudo,
        num_boost_round=3000,
        nfold=5,
        early_stopping_rounds=100,
        verbose_eval=False
    )

    best_rmse = model["test-rmse-mean"].iloc[-1]
    return best_rmse

In [9]:
# Features and target
# X, y = data.data, data.target
X = data.drop(columns=["MedHouseVal"])
y = data["MedHouseVal"]

# Train/test split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset Shape: {X.shape}")
print(f"Average Target: {np.mean(y):.2f} (hundreds of k$)")

# ==========================================
# 2. Baseline (Without Tuning)
# ==========================================
print("\n--- Training Baseline Model (Default) ---")
dtrain_full = xgb.DMatrix(X_train_full, label=y_train_full)
dtest = xgb.DMatrix(X_test, label=y_test)

# Default parameters
params_base = {
    'objective': 'reg:squarederror',  # Regression
    'tree_method': 'hist',
    'random_state': 42
}

model_base = xgb.train(params_base, dtrain_full, num_boost_round=100)
preds_base = model_base.predict(dtest)

# Compute Baseline RMSE
rmse_base = np.sqrt(mean_squared_error(y_test, preds_base))
print(f"Baseline RMSE: {rmse_base:.4f}")

# ==========================================
# 2.5. Pseudo-Labeling
# ==========================================
X_unlabeled = X.sample(200, replace=False).copy()
# Pretend we don't know MedHouseVal
# X_unlabeled = df_unlabeled[features]  # <-- for real use

# Scale unlabeled data
X_unl_scaled = scaler.transform(X_unlabeled)

# Predict pseudo-labels using the baseline model
d_unl = xgb.DMatrix(X_unlabeled)
pseudo_labels = model_base.predict(d_unl)

print(f"Created {len(pseudo_labels)} pseudo-labels")

# Build extended dataset
X_pseudo = pd.concat([X_train_full, X_unlabeled], axis=0)
y_pseudo = pd.concat([y_train_full, pd.Series(pseudo_labels, index=X_unlabeled.index)])

print("Extended dataset ready:")
print(f" - Original: {len(X_train_full)}")
print(f" - Pseudo-labels: {len(X_unlabeled)}")
print(f" - Total: {len(X_pseudo)}")

# Extended DMatrix
dtrain_pseudo = xgb.DMatrix(X_pseudo, label=y_pseudo)

# ==========================================
# 3. Optimization with Optuna
# ==========================================
print("\n--- Starting Optuna Tuning ---")

# Create study (minimize RMSE)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=200, timeout=600)

print(f"\nBest RMSE found by Optuna (Validation): {study.best_value:.4f}")
print("Best parameters:", study.best_params)

# ==========================================
# 4. Final Model Training (Best Params + Shrinkage)
# ==========================================
print("\n--- Training Final Optimized Model ---")

best_params = study.best_params
best_params['objective'] = 'reg:squarederror'
best_params['eval_metric'] = 'rmse'
best_params['tree_method'] = 'hist'

# "SHRINKAGE" STRATEGY (Refine gradient descent)
# Reduce the learning rate found for finer final precision
best_params['learning_rate'] = best_params['learning_rate'] * 0.5
# Increase number of trees accordingly (high safety buffer, early stopping will manage)
num_round_final = 5000

model_opt = xgb.train(
    best_params, 
    dtrain_pseudo,
    num_boost_round=num_round_final,
    evals=[(dtest, "test")],  # Test set used ONLY to stop training at the right point
    early_stopping_rounds=100,
    verbose_eval=False
)

# ==========================================
# 5. Results Analysis
# ==========================================
print("\n--- FINAL COMPARISON ---")

preds_opt = model_opt.predict(dtest)
rmse_opt = np.sqrt(mean_squared_error(y_test, preds_opt))

# R2 Score (coefficient of determination)
r2_base = r2_score(y_test, preds_base)
r2_opt = r2_score(y_test, preds_opt)

print(f"Baseline RMSE:    {rmse_base:.4f}")
print(f"Optimized RMSE:   {rmse_opt:.4f}")
print(f"--> Error Improvement: {rmse_base - rmse_opt:.4f} (Lower is better)")
print("-" * 30)
print(f"Baseline R2 Score:    {r2_base:.4f}")
print(f"Optimized R2 Score:   {r2_opt:.4f}")
print(f"--> Extra Variance Explained: +{(r2_opt - r2_base)*100:.2f}%")

Dataset Shape: (13561, 11)
Average Target: 1.85 (hundreds of k$)

--- Training Baseline Model (Default) ---


[I 2026-03-27 22:35:09,035] A new study created in memory with name: no-name-9d6b32cb-4c44-4d2c-88bb-f2862050880c


Baseline RMSE: 0.3548
Created 200 pseudo-labels
Extended dataset ready:
 - Original: 10848
 - Pseudo-labels: 200
 - Total: 11048

--- Starting Optuna Tuning ---


[I 2026-03-27 22:35:10,972] Trial 0 finished with value: 0.3781802981687996 and parameters: {'max_depth': 12, 'min_child_weight': 23, 'gamma': 0.07551026367865302, 'subsample': 0.8240412812988416, 'colsample_bytree': 0.907080660434388, 'lambda': 7.224041437061901, 'alpha': 9.453189248379909e-08, 'learning_rate': 0.34120863877195234}. Best is trial 0 with value: 0.3781802981687996.
[I 2026-03-27 22:35:12,512] Trial 1 finished with value: 0.3897406828717559 and parameters: {'max_depth': 11, 'min_child_weight': 19, 'gamma': 0.48276068631884306, 'subsample': 0.6867607884317334, 'colsample_bytree': 0.6940122896294212, 'lambda': 0.06704918902257749, 'alpha': 0.06766482292062145, 'learning_rate': 0.4440659497982956}. Best is trial 0 with value: 0.3781802981687996.
[I 2026-03-27 22:35:26,038] Trial 2 finished with value: 0.3526289997813649 and parameters: {'max_depth': 13, 'min_child_weight': 7, 'gamma': 0.003053818680108773, 'subsample': 0.8585735211596608, 'colsample_bytree': 0.8761130204760


Best RMSE found by Optuna (Validation): 0.3413
Best parameters: {'max_depth': 17, 'min_child_weight': 22, 'gamma': 3.1099363274778763e-06, 'subsample': 0.6819223189571961, 'colsample_bytree': 0.7581152440722306, 'lambda': 1.471738508217411, 'alpha': 0.12070654769713053, 'learning_rate': 0.011929634558025468}

--- Training Final Optimized Model ---

--- FINAL COMPARISON ---
Baseline RMSE:    0.3548
Optimized RMSE:   0.3334
--> Error Improvement: 0.0214 (Lower is better)
------------------------------
Baseline R2 Score:    0.8194
Optimized R2 Score:   0.8406
--> Extra Variance Explained: +2.11%


In [10]:
def objective(trial):

    params = {
        # --- Hardware Config ---
        'device_type': 'cpu',         # Use CPU (not GPU)
        'n_jobs': -1,                 # Use all available cores
        'verbose': -1,                # Suppress logs

        # --- Speed / Efficiency ---
        'max_bin': 255,               # Number of bins for histogram-based splits
        'bagging_freq': 1,            # Frequency of bagging
        'bagging_fraction': 0.8,      # Fraction of data used in each iteration

        # --- Model Parameters ---
        'objective': 'regression',    # Task: regression
        'metric': 'rmse',             # Metric: RMSE
        'boosting_type': 'gbdt',      # Gradient Boosted Decision Trees
        
        # --- Search Space (Reduced for faster convergence) ---
        'n_estimators': 1000,         # Maximum number of boosting rounds
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.3, log=True),  # LR: higher = faster learning
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),      # Fewer leaves = simpler trees
        'max_depth': trial.suggest_int('max_depth', 3, 8),           # Shallower trees
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 100, 1000),  # Minimum samples per leaf
        
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),    # L1 regularization
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),  # L2 regularization
    }

    # Create the LightGBM model
    model = lgb.LGBMRegressor(**params)
    
    # Early stopping / pruning callback for Optuna
    pruning_callback = LightGBMPruningCallback(trial, "rmse")

    # Train model
    model.fit(
        X_train_full, y_train_full,
        eval_set=[(X_test, y_test)],  # Validation set
        eval_metric='rmse',
        callbacks=[
            # Stop early if no improvement for 10 rounds (aggressive)
            lgb.early_stopping(stopping_rounds=10, verbose=False),
            pruning_callback
        ]
    )

    # Predict on test set and compute RMSE
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    # Return RMSE for Optuna to minimize
    return rmse

In [11]:
# Split features and target
# X, y = data.data, data.target
X = data.drop(columns=["MedHouseVal"])  # Features (all columns except "MedHouseVal")
y = data["MedHouseVal"]                # Target variable ("MedHouseVal")

# Split the dataset into training and test sets (80% train, 20% test)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)  # Fit and transform training data
X_test_scaled = scaler.transform(X_test)             # Transform test data

print(f"Dataset Shape: {X.shape}")
print(f"Average Target: {np.mean(y):.2f} (hundreds of $K)")

# ==========================================
# 2. Baseline Model (No Tuning)
# ==========================================
dtrain_full = lgb.Dataset(X_train_full, label=y_train_full)
dtest = lgb.Dataset(X_test, label=y_test)

# Baseline LightGBM parameters
params_base = {
    'objective': 'regression',
    'metric': 'rmse',
    'seed': 42
}

# Train baseline model
model_base = lgb.train(params_base, dtrain_full, num_boost_round=100)

# Make predictions with baseline model
preds_base = model_base.predict(X_test)

# Calculate RMSE for baseline
rmse_base = np.sqrt(mean_squared_error(y_test, preds_base))
print(f"Baseline RMSE: {rmse_base:.4f}")

# ==========================================
# 3. Hyperparameter Optimization with Optuna
# ==========================================
print("\n--- Starting Optuna Tuning ---")

# Create a study to minimize RMSE
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=200, timeout=600)  # Run optimization

print(f"\nBest Validation RMSE found by Optuna: {study.best_value:.4f}")
print("Best parameters:", study.best_params)

# ==========================================
# 4. Train Final Model (Best Params + Shrinkage)
# ==========================================
print("\n--- Training Optimized Final Model ---")

best_params = study.best_params
best_params['objective'] = 'regression'
best_params['eval_metric'] = 'rmse'
best_params['tree_method'] = 'hist'

# "Shrinkage" strategy: lower learning rate for finer convergence
best_params['learning_rate'] = best_params['learning_rate'] * 0.5
num_round_final = 5000  # Increase number of boosting rounds accordingly

model_opt = lgb.train(
    best_params,
    dtrain_full,
    num_boost_round=num_round_final,
    valid_sets=[dtest],
    valid_names=["test"],
    callbacks=[lgb.early_stopping(stopping_rounds=100)]  # Stop if no improvement
)

# ==========================================
# 5. Results Analysis
# ==========================================
print("\n--- FINAL COMPARISON ---")

# Predictions with optimized model
preds_opt = model_opt.predict(X_test)
rmse_opt = np.sqrt(mean_squared_error(y_test, preds_opt))

# R2 score (coefficient of determination)
r2_base = r2_score(y_test, preds_base)
r2_opt = r2_score(y_test, preds_opt)

print(f"Baseline RMSE:    {rmse_base:.4f}")
print(f"Optimized RMSE:   {rmse_opt:.4f}")
print(f"--> Error Improvement: {rmse_base - rmse_opt:.4f} (Lower is better)")
print("-" * 30)
print(f"Baseline R2 Score:    {r2_base:.4f}")
print(f"Optimized R2 Score:   {r2_opt:.4f}")
print(f"--> Additional Explained Variance: +{(r2_opt - r2_base)*100:.2f}%")

Dataset Shape: (13561, 11)
Average Target: 1.85 (hundreds of $K)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000444 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2602
[LightGBM] [Info] Number of data points in the train set: 10848, number of used features: 11
[LightGBM] [Info] Start training from score 1.848509


[I 2026-03-27 22:45:30,028] A new study created in memory with name: no-name-a53f5d18-7b24-496c-9c55-9e72dbf584d5


Baseline RMSE: 0.3491

--- Starting Optuna Tuning ---


[I 2026-03-27 22:45:30,329] Trial 0 finished with value: 0.3672858520982726 and parameters: {'learning_rate': 0.2818214945882821, 'num_leaves': 63, 'max_depth': 6, 'min_data_in_leaf': 368, 'reg_alpha': 0.01302319858185117, 'reg_lambda': 0.0019159652273788717}. Best is trial 0 with value: 0.3672858520982726.
[I 2026-03-27 22:45:30,845] Trial 1 finished with value: 0.36198468757086405 and parameters: {'learning_rate': 0.17776189059132685, 'num_leaves': 46, 'max_depth': 5, 'min_data_in_leaf': 315, 'reg_alpha': 1.6662791576957319, 'reg_lambda': 0.2280726297550005}. Best is trial 1 with value: 0.36198468757086405.
[I 2026-03-27 22:45:31,926] Trial 2 finished with value: 0.34768373184383067 and parameters: {'learning_rate': 0.16770195156484463, 'num_leaves': 76, 'max_depth': 7, 'min_data_in_leaf': 123, 'reg_alpha': 4.239923469018183, 'reg_lambda': 5.536377543328922}. Best is trial 2 with value: 0.34768373184383067.
[I 2026-03-27 22:45:33,256] Trial 3 finished with value: 0.3519955640122232 a


Best Validation RMSE found by Optuna: 0.3477
Best parameters: {'learning_rate': 0.16770195156484463, 'num_leaves': 76, 'max_depth': 7, 'min_data_in_leaf': 123, 'reg_alpha': 4.239923469018183, 'reg_lambda': 5.536377543328922}

--- Training Optimized Final Model ---
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[828]	test's l2: 0.114833

--- FINAL COMPARISON ---
Baseline RMSE:    0.3491
Optimized RMSE:   0.3389
--> Error Improvement: 0.0102 (Lower is better)
------------------------------
Baseline R2 Score:    0.8252
Optimized R2 Score:   0.8353
--> Additional Explained Variance: +1.01%


**RESULTS**

Baseline RMSE:    0.4718
Optimized RMSE: 0.4300

param = {
        ‘verbosity’: 0,
        ‘objective’: ‘reg:squarederror’,
        ‘eval_metric’: ‘rmse’, # Metrics for early stopping and pruning
        ‘tree_method’: ‘hist’,
        
        # --- Structure ---
        # California has complex interactions; let’s allow deeper trees
        ‘max_depth’: trial.suggest_int(‘max_depth’, 2, 15),
        # Important to avoid overfitting on price outliers
        'min_child_weight': trial.suggest_int(‘min_child_weight’, 1, 20),
        ‘gamma’: trial.suggest_float(‘gamma’, 1e-8, 1.0, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float(‘subsample’, 0.6, 1.0),
        ‘colsample_bytree’: trial.suggest_float(‘colsample_bytree’, 0.6, 1.0),
        
        # --- Regularization ---
        # Essential in regression to avoid chasing extreme prices
        ‘lambda’: trial.suggest_float(‘lambda’, 1e-9, 25.0, log=True),
        ‘alpha’: trial.suggest_float(‘alpha’, 1e-9, 25.0, log=True),
        
        # -- - Learning ---
        ‘learning_rate’: trial.suggest_float(‘learning_rate’, 0.01, 0.3, log=True),
    }

study.optimize(objective, n_trials=70, timeout=600)

=============================================================================== <br>
Baseline RMSE:    0.4718
Optimized RMSE: 0.4310

with scaler
param = {
        ‘verbosity’: 0,
        ‘objective’: ‘reg:squarederror’,
        ‘eval_metric’: ‘rmse’, # Metric for early stopping and pruning
        ‘tree_method’: ‘hist’,
        
        # --- Structure ---
        # California has complex interactions; let's allow deeper trees
        ‘max_depth’: trial.suggest_int(‘max_depth’, 2, 20),
        # Important to avoid overfitting on price outliers
        ‘min_child_weight’: trial.suggest_int(‘min_child_weight’, 1, 25),
        'gamma': trial.suggest_float(‘gamma’, 1e-9, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float(‘subsample’, 0.6, 1.0),
        ‘colsample_bytree’: trial.suggest_float(‘colsample_bytree’, 0.6, 1.0),
        
        # --- Regularization ---
        # Essential in regression to avoid chasing extreme prices
        ‘lambda’: trial.suggest_float(‘lambda’, 1e-6, 20.0, log=True),
        ‘alpha’: trial.suggest_float(‘alpha’, 1e-6, 20.0, log=True),
        
        # -- - Learning ---
        ‘learning_rate’: trial.suggest_float(‘learning_rate’, 0.01, 0.8),
    }

study.optimize(objective, n_trials=200, timeout=600)

=============================================================================== <br>

Baseline RMSE:    0.3522
Optimized RMSE: 0.3379

with scaler

study.optimize(objective, n_trials=200, timeout=600)

param = {
        ‘verbosity’: 0,
        ‘objective’: ‘reg:squarederror’,
        ‘eval_metric’: ‘rmse’, # Metric for early stopping and pruning
        ‘tree_method’: ‘hist’,
        
        # --- Structure ---
        # California has complex interactions; let's allow deeper trees
        ‘max_depth’: trial.suggest_int(‘max_depth’, 2, 20),
        # Important to avoid overfitting on price outliers
        ‘min_child_weight’: trial.suggest_int(‘min_child_weight’, 1, 25),
        'gamma': trial.suggest_float(‘gamma’, 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float(‘subsample’, 0.6, 1.0),
        ‘colsample_bytree’: trial.suggest_float(‘colsample_bytree’, 0.6, 1.0),
        
        # --- Regularization ---
        # Essential in regression to avoid chasing extreme prices
        ‘lambda’: trial.suggest_float(‘lambda’, 1e-9, 15.0, log=True),
        ‘alpha’: trial.suggest_float(‘alpha’, 1e-9, 15.0, log=True),
        
        # -- - Learning ---
        ‘learning_rate’: trial.suggest_float(‘learning_rate’, 0.001, 0.6),
    }

# removing outliers
numeric_cols = data.select_dtypes(include=[“float”, “int”]).columns.tolist()

for col in numeric_cols:
    Q1 = data[col].quantile(0.3)
    Q3 = data[col].quantile(0.7)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    data[col] = np.where(data[col].between(lower, upper), data[col], np.nan)

data = data.dropna()

#combined features
data[‘Ave_Room_Bed’] = data[‘AveRooms’] / data[‘AveBedrms’]
data[‘HouseRooms’] = data[‘HouseAge’] / data[‘AveRooms’]

Best RMSE found by Optuna (Validation): 0.3544
Best parameters: {‘max_depth’: 13, ‘min_child_weight’: 23, ‘gamma’: 3.8838420870962745e-06, ‘subsample’: 0.7570628531701007, ‘colsample_bytree’: 0.9109736816364161, ‘lambda’: 0.02307771384295654, ‘alpha’: 0.9454327224813569, ‘learning_rate’: 0.06885420667941514}

=============================================================================== <br>
Baseline RMSE:    0.3548
Optimized RMSE: 0.3297

with scaler

study.optimize(objective, n_trials=200, timeout=600)

param = {
        ‘verbosity’: 0,
        ‘objective’: ‘reg:squarederror’,
        ‘eval_metric’: ‘rmse’, # Metric for early stopping and pruning
        ‘tree_method’: ‘hist’,
        
        # --- Structure ---
        # California has complex interactions; let's allow deeper trees
        ‘max_depth’: trial.suggest_int(‘max_depth’, 2, 20),
        # Important to avoid overfitting on price outliers
        ‘min_child_weight’: trial.suggest_int(‘min_child_weight’, 1, 25),
        'gamma': trial.suggest_float(‘gamma’, 1e-7, 1.5, log=True),
        
        # --- Randomness ---
        'subsample': trial.suggest_float(‘subsample’, 0.6, 1.0),
        ‘colsample_bytree’: trial.suggest_float(‘colsample_bytree’, 0.6, 1.0),
        
        # --- Regularization ---
        # Essential in regression to avoid chasing extreme prices
                'lambda': trial.suggest_float(‘lambda’, 1e-9, 15.0, log=True),
        ‘alpha’: trial.suggest_float(‘alpha’, 1e-9, 15.0, log=True),
        
        # --- Learning ---
        'learning_rate': trial.suggest_float(‘learning_rate’, 0.001, 0.6),
    }

# removing outliers
numeric_cols = data.select_dtypes(include=[“float”, “int”]).columns.tolist()

for col in numeric_cols:
    Q1 = data[col].quantile(0.3)
    Q3 = data[col].quantile(0.7)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    data[col] = np.where(data[col].between(lower, upper), data[col], np.nan)

data = data.dropna()

#combined features
data[‘Ave_Room_Bed’] = data[‘AveRooms’] / data[‘AveBedrms’]
data[‘HouseRooms’] = data[‘HouseAge’] / data[‘AveRooms’]
data[‘Rooms_per_Person’] = data[‘AveRooms’] / data[‘AveOccup’]

Best RMSE found by Optuna (Validation): 0.3490
Best parameters: {‘max_depth’: 6, ‘min_child_weight’: 10, ‘gamma’: 5.35007610966066e-05, ‘subsample’: 0.8096500866788845, ‘colsample_bytree’: 0.7174314936237725, ‘lambda’: 2.180030783832174e-06, ‘alpha’: 9.101912061894957e-09, ‘learning_rate’: 0.042067598298740405}